# 02 — Análise comparativa do RGF

Rankings, grupos, score fiscal, destaques e gráficos do Poder Executivo estadual. O score combina nível, margens, recorrência, tendência e volatilidade; consulte o README para a fórmula completa.

In [ ]:
from pathlib import Path
import sys, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from rgf.analysis import build_indicators, annual_ranking
from rgf.charts import generate_charts
from rgf.export import executive_summary, export_excel
base = pd.read_csv(ROOT / 'dados/rgf_estados_2015_2025_tratado.csv')
indicadores = build_indicators(base)
ranking = annual_ranking(base)

## Ranking anual e consolidado

In [ ]:
ultimo_ano = int(base.dropna(subset=['DTP_RCL']).ano.max())
display(ranking.query('ano == @ultimo_ano'))
display(indicadores[['ranking_historico','UF','nome_estado','score_fiscal','grupo']])

## Destaques

In [ ]:
destaques = {
 '5 melhores': indicadores.nsmallest(5, 'ranking_historico'),
 '5 piores': indicadores.nlargest(5, 'ranking_historico'),
 'mais melhoraram': indicadores.nsmallest(5, 'variacao_primeiro_ultimo_pp'),
 'mais pioraram': indicadores.nlargest(5, 'variacao_primeiro_ultimo_pp'),
 'maior recorrência': indicadores.sort_values(['anos_acima_maximo','anos_acima_prudencial','anos_acima_alerta'], ascending=False).head(5),
 'maior volatilidade': indicadores.nlargest(5, 'volatilidade_desvio_padrao'),
 'deterioração recente': indicadores.nlargest(5, 'tendencia_recente_pp_ano'),
 'melhora recente': indicadores.nsmallest(5, 'tendencia_recente_pp_ano'),
}
for titulo, tabela in destaques.items():
    print('\n', titulo.upper()); display(tabela)

## Gráficos executivos

In [ ]:
generate_charts(base, indicadores)
from IPython.display import Image, display
for arquivo in sorted((ROOT / 'graficos').glob('*.png')):
    print(arquivo.stem); display(Image(filename=str(arquivo), width=900))

## Panorama da Despesa com Pessoal dos Estados Brasileiros – RGF 2015–2025

In [ ]:
sintese = executive_summary(base, indicadores)
print(sintese)
export_excel(base, indicadores, ranking)